# Chapter 25 Guided To-Do Practice Labs

**Book Reference:** Chapter 25, Section 25.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 unsupervised learning functions (K-Means from scratch, K-sweep evaluation, Ward hierarchical clustering, DBSCAN anomaly flagging, SVD-based PCA, and variance-target component selection), and pass each **function name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch25_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import sharukh_tutor as tutor

tutor.activate(chapter="ch25", notebook="guided_todos")


---
## To-Do 25.1 (Pure-NumPy K-Means): `kmeans_from_scratch(X, k, max_iter=50, seed=42)`

Implement pure-NumPy iterative K-Means without mutating `X`:
- Use `np.random.default_rng(seed)` to select `k` distinct initial rows of `X` as centroids (`replace=False`).
- Alternate nearest-centroid Euclidean assignment and centroid mean recomputation (keeping a centroid unchanged if empty) until convergence or `max_iter`.
- Return `(labels, centroids, inertia)` where `labels` is a 1D integer array of shape `(n,)`, `centroids` has shape `(k, d)`, and `inertia` is a Python `float` equal to the total squared Euclidean distance to assigned centroids.

In [ ]:
def kmeans_from_scratch(X, k, max_iter=50, seed=42):
    """Run pure-NumPy K-Means and return (labels, centroids, inertia)."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    X_arr = np.asarray(X, dtype=float)
    rng = np.random.default_rng(seed)
    init_idx = rng.choice(len(X_arr), size=k, replace=False)
    centroids = X_arr[init_idx].copy()
    for _ in range(max_iter):
        dists = ((X_arr[:, None, :] - centroids[None, :, :]) ** 2)
        labels = np.argmin(dists.sum(axis=2), axis=1).astype(int)
        new_c = centroids.copy()
        for cid in range(k):
            mask = labels == cid
            if np.any(mask):
                new_c[cid] = X_arr[mask].mean(axis=0)
        if np.allclose(new_c, centroids):
            centroids = new_c
            break
        centroids = new_c
    dists = ((X_arr[:, None, :] - centroids[None, :, :]) ** 2)
    labels = np.argmin(dists.sum(axis=2), axis=1).astype(int)
    inertia = float(np.sum((X_arr - centroids[labels]) ** 2))
    return labels, centroids, inertia
    ### END SOLUTION


In [ ]:
tutor.check("ch25_todo_01", kmeans_from_scratch)


---
## To-Do 25.2 (Elbow & Silhouette Sweep): `evaluate_k_range(X_scaled, k_values, seed=42)`

Sweep each integer `k` in `k_values` using `KMeans(n_clusters=k, n_init=10, random_state=seed)`:
- Compute `inertia` (`km.inertia_`) and `silhouette` (`silhouette_score(X_scaled, labels)`), both rounded to **4 decimals**.
- Return a `pd.DataFrame` with exact columns `["k", "inertia", "silhouette"]`.

In [ ]:
def evaluate_k_range(X_scaled, k_values, seed=42):
    """Sweep k_values with KMeans and return inertia/silhouette table."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    rows = []
    for k in k_values:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, labels)
        rows.append({
            "k": int(k),
            "inertia": round(float(km.inertia_), 4),
            "silhouette": round(float(sil), 4),
        })
    return pd.DataFrame(rows)
    ### END SOLUTION


In [ ]:
tutor.check("ch25_todo_02", evaluate_k_range)


---
## To-Do 25.3 (Hierarchical Ward Clustering): `fit_agglomerative_ward(X_scaled, n_clusters=3)`

Fit `AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")` on `X_scaled`:
- Return `{"labels": labels, "silhouette": round4, "cluster_sizes": {int_label: int_count}}`.

In [ ]:
def fit_agglomerative_ward(X_scaled, n_clusters=3):
    """Fit Ward agglomerative clustering and summarize silhouette/sizes."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    model = AgglomerativeClustering(
        n_clusters=n_clusters, linkage="ward"
    )
    labels = model.fit_predict(X_scaled)
    unique, counts = np.unique(labels, return_counts=True)
    sizes = {int(c): int(n) for c, n in zip(unique, counts)}
    sil = round(float(silhouette_score(X_scaled, labels)), 4)
    return {
        "labels": labels,
        "silhouette": sil,
        "cluster_sizes": sizes,
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch25_todo_03", fit_agglomerative_ward)


---
## To-Do 25.4 (Density Outlier Detection): `detect_dbscan_anomalies(X_scaled, eps=0.8, min_samples=5)`

Fit `DBSCAN(eps=eps, min_samples=min_samples)` on `X_scaled`:
- Count `n_clusters` excluding noise label `-1` and count `n_noise` where `labels == -1`.
- Return `{"n_clusters": int, "n_noise": int, "labels": labels}`.

In [ ]:
def detect_dbscan_anomalies(X_scaled, eps=0.8, min_samples=5):
    """Run DBSCAN and separate dense cluster count from -1 noise count."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    model = DBSCAN(eps=eps, min_samples=min_samples)
    labels = model.fit_predict(X_scaled)
    n_clusters = int(len(set(labels.tolist()) - {-1}))
    n_noise = int(np.sum(labels == -1))
    return {
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "labels": labels,
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch25_todo_04", detect_dbscan_anomalies)


---
## To-Do 25.5 (PCA via NumPy SVD): `pca_from_scratch_svd(X, n_components=2)`

Implement PCA from scratch without mutating `X`:
- Mean-center `X` by subtracting each column mean (`X_c = X_arr - X_arr.mean(axis=0)`).
- Compute `U, S, Vt = np.linalg.svd(X_c, full_matrices=False)`.
- Return `(X_projected, explained_variance_ratio)` of shapes `(n, n_components)` and `(n_components,)`.

In [ ]:
def pca_from_scratch_svd(X, n_components=2):
    """Center X, run SVD, and return (X_projected, explained_var_ratio)."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    X_arr = np.asarray(X, dtype=float)
    X_c = X_arr - X_arr.mean(axis=0)
    _, S, Vt = np.linalg.svd(X_c, full_matrices=False)
    X_proj = X_c @ Vt[:n_components].T
    evr = (S ** 2) / np.sum(S ** 2)
    return X_proj, evr[:n_components]
    ### END SOLUTION


In [ ]:
tutor.check("ch25_todo_05", pca_from_scratch_svd)


---
## To-Do 25.6 (Variance Target Selector): `components_for_variance_target(X_scaled, target_var=0.90)`

Fit a full `PCA()` on `X_scaled` and return the smallest Python `int` `k` such that `np.cumsum(pca.explained_variance_ratio_)[k - 1] >= target_var`.

In [ ]:
def components_for_variance_target(X_scaled, target_var=0.90):
    """Return smallest int k reaching target_var cumulative variance."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    pca = PCA().fit(X_scaled)
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    return int(np.argmax(cum_var >= target_var) + 1)
    ### END SOLUTION


In [ ]:
tutor.check("ch25_todo_06", components_for_variance_target)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 25.1 (Empty-Cluster Guard & Final Assignment):** If a centroid receives zero points on an iteration, computing `X_arr[mask].mean(axis=0)` produces `NaN`s that poison every subsequent distance calculation. Checking `if np.any(mask)` and recomputing final `labels` against the converged `centroids` guarantees finite, consistent SSE inertia.
- **To-Do 25.4 (Noise Label `-1` in DBSCAN):** `len(set(labels))` mistakenly counts `-1` (noise points) as a real cluster! Always subtract `{-1}` so all-noise runs cleanly report `n_clusters = 0`.
- **To-Do 25.5 & 25.6 (Mean-Centering & Off-by-One Indexing):** Skipping `X_arr - X_arr.mean(axis=0)` before `np.linalg.svd` causes PC1 to point toward the global mean offset rather than the direction of maximum variance. Similarly, `np.argmax(cum_var >= target_var)` returns a 0-based index, so you must add `1` to obtain the 1-indexed component count `k`.

In [ ]:
tutor.summary()
